# Delta table Query util
DuckDB natively supports reading Delta tables using `delta_scan()`.

In [1]:
import os
import re

import certifi
import duckdb

from electricity_maps.config import get_settings
import pandas as pd

os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()
pd.set_option('display.max_columns', None)

get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir


In [2]:
def get_sql_df(sql_query):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs; INSTALL aws; LOAD aws;")
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET aws_s3 (TYPE S3, KEY_ID '{settings.aws_access_key_id}', SECRET '{settings.aws_secret_access_key}', REGION '{settings.aws_region}');")

    def replace_table(match):
        keyword = match.group(1)
        schema = match.group(2)
        table = match.group(3)

        if schema == "bronze":
            return f"{keyword} read_parquet('{DATA_DIR}/{schema}/{table}/**/*.parquet')"
        else:
            return f"{keyword} delta_scan('{DATA_DIR}/{schema}/{table}')"

    parsed_query = re.sub(
        r'(?i)(FROM|JOIN)\s+([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)',
        replace_table,
        sql_query
    )
    return con.execute(parsed_query).df()

def print_sql_df(sql_query):
    df = get_sql_df(sql_query)
    return df

In [3]:
print("Bronze Daily Flows Summary:")
print_sql_df('select * from bronze.electricity_flows order by process_ts desc limit 10')

Bronze Daily Flows Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,day,month,year
0,1777205939689,2026-04-26 17:48:59.689627+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-26 14:30:00+05:30,2026-04-26 17:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",3,26,04,2026
1,1777205502572,2026-04-26 17:41:42.572706+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-25 17:30:00+05:30,2026-04-26 17:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,26,04,2026


In [4]:
print("Bronze Daily Mix Summary:")
print_sql_df('select * from bronze.electricity_mix order by process_ts desc limit 10')

Bronze Daily Mix Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,day,month,year
0,1777205939689,2026-04-26 17:48:59.689627+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-26 14:30:00+05:30,2026-04-26 17:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",3,26,04,2026
1,1777205502572,2026-04-26 17:41:42.572706+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-25 17:30:00+05:30,2026-04-26 17:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,26,04,2026


In [5]:
print("Silver :")
print_sql_df('select * from silver.electricity_mix order by process_ts desc limit 10')

Silver :


,process_ts,zone,datetime,updated_at,is_estimated,estimation_method,nuclear_mw,geothermal_mw,biomass_mw,coal_mw,wind_mw,solar_mw,hydro_mw,gas_mw,oil_mw,unknown_mw,hydro_storage_charge_mw,hydro_storage_discharge_mw,battery_storage_charge_mw,battery_storage_discharge_mw,flow_exports_mw,flow_imports_mw,year,month,day
0,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,True,SANDBOX_MODE_DATA,24184.452,NaN,934.787,0.0,670.105,9236.720,2752.129,329.130,40.678,NaN,2092.978,NaN,6.517,NaN,8865.0,1151.0,2026,4,26
1,1777206207772,FR,2026-04-26 10:00:00,2026-04-26 11:54:21.220,True,SANDBOX_MODE_DATA,34293.006,NaN,1428.390,0.0,501.067,11024.213,2877.292,390.610,37.977,NaN,2737.183,NaN,26.680,NaN,6404.0,1660.0,2026,4,26
2,1777206207772,FR,2026-04-26 11:00:00,2026-04-26 11:54:21.220,True,SANDBOX_MODE_DATA,27653.723,NaN,1071.846,0.0,560.688,9640.912,3118.090,435.373,43.521,NaN,2834.244,NaN,13.286,NaN,7922.0,1082.0,2026,4,26
3,1777205520389,FR,2026-04-26 00:00:00,2026-04-26 02:10:01.513,True,SANDBOX_MODE_DATA,46622.317,NaN,1250.727,0.0,3616.659,0.000,4731.153,337.941,42.136,NaN,NaN,1756.516,NaN,8.413,16564.0,0.0,2026,4,26
4,1777205520389,FR,2026-04-26 01:00:00,2026-04-26 03:08:35.425,True,SANDBOX_MODE_DATA,40526.998,NaN,1053.134,0.0,3445.310,0.000,4582.203,408.801,31.615,NaN,NaN,552.478,65.462,NaN,18025.0,0.0,2026,4,26
5,1777205520389,FR,2026-04-26 02:00:00,2026-04-26 04:25:02.150,True,SANDBOX_MODE_DATA,49071.986,NaN,1151.768,0.0,2777.404,0.000,5139.354,473.772,43.217,NaN,NaN,269.315,14.201,NaN,20324.0,0.0,2026,4,26
6,1777205520389,FR,2026-04-26 06:00:00,2026-04-26 08:23:43.965,True,SANDBOX_MODE_DATA,44138.299,NaN,1065.250,0.0,3404.097,3037.587,3381.583,279.081,37.309,NaN,1013.599,NaN,NaN,24.063,16529.0,0.0,2026,4,26
7,1777205520389,FR,2026-04-26 05:00:00,2026-04-26 07:23:42.971,True,SANDBOX_MODE_DATA,47082.073,NaN,1375.444,0.0,3485.157,760.923,4155.831,434.822,33.764,NaN,452.599,NaN,20.537,NaN,17491.0,0.0,2026,4,26
8,1777205520389,FR,2026-04-26 04:00:00,2026-04-26 06:23:48.333,True,SANDBOX_MODE_DATA,38408.712,NaN,948.866,0.0,3620.826,125.026,5450.484,295.858,35.171,NaN,NaN,292.415,NaN,19.498,17459.0,0.0,2026,4,26
9,1777205520389,FR,2026-04-26 03:00:00,2026-04-26 05:08:26.763,True,SANDBOX_MODE_DATA,39383.117,NaN,1174.848,0.0,4254.088,0.000,4581.383,454.518,28.349,NaN,NaN,315.080,NaN,30.827,14858.0,0.0,2026,4,26


In [6]:
print("Silver :")
print_sql_df('select * from silver.electricity_flows order by process_ts desc limit 10')

Silver :


,process_ts,zone,datetime,updated_at,neighbor_zone,direction,power_mw,year,month,day
0,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,BE,import,1922.0,2026,4,26
1,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,CH,export,835.0,2026,4,26
2,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,DE,import,389.0,2026,4,26
3,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,ES,import,2612.0,2026,4,26
4,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,GB,export,3015.0,2026,4,26
5,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,IT-NO,import,216.0,2026,4,26
6,1777206207772,FR,2026-04-26 09:00:00,2026-04-26 11:54:21.220,LU,export,135.0,2026,4,26
7,1777206207772,FR,2026-04-26 10:00:00,2026-04-26 11:54:21.220,BE,import,868.0,2026,4,26
8,1777206207772,FR,2026-04-26 10:00:00,2026-04-26 11:54:21.220,CH,export,1085.0,2026,4,26
9,1777206207772,FR,2026-04-26 10:00:00,2026-04-26 11:54:21.220,DE,import,299.0,2026,4,26


In [7]:
print("Gold Daily Exports:")
print_sql_df('select * from gold.daily_exports order by process_ts desc limit 10')

Gold Daily Exports:


,process_ts,zone,zone_name,destination_zone,destination_zone_name,date,export_mwh,hours_covered,year,month,day
0,1777206222712,FR,France,IT-NO,Italy North,2026-04-26,11469.0,12,2026,4,26
1,1777206222712,FR,France,GB,Great Britain,2026-04-26,36309.0,12,2026,4,26
2,1777206222712,FR,France,LU,LU,2026-04-26,2354.0,12,2026,4,26
3,1777205541513,FR,France,IT-NO,Italy North,2026-04-25,26146.0,12,2026,4,25
4,1777205541513,FR,France,GB,Great Britain,2026-04-25,36655.0,12,2026,4,25
5,1777205541513,FR,France,LU,LU,2026-04-25,2315.0,12,2026,4,25


In [8]:
print("\nGold Daily Imports:")
print_sql_df("SELECT * FROM gold.daily_imports order by process_ts desc LIMIT 10")


Gold Daily Imports:


,process_ts,zone,zone_name,source_zone,source_zone_name,date,import_mwh,hours_covered,year,month,day
0,1777206222712,FR,France,CH,Switzerland,2026-04-26,8225.0,12,2026,4,26
1,1777206222712,FR,France,ES,Spain,2026-04-26,25734.0,12,2026,4,26
2,1777206222712,FR,France,BE,Belgium,2026-04-26,44191.0,12,2026,4,26
3,1777206222712,FR,France,DE,Germany,2026-04-26,19320.0,12,2026,4,26
4,1777205541513,FR,France,ES,Spain,2026-04-25,28389.0,12,2026,4,25
5,1777205541513,FR,France,BE,Belgium,2026-04-25,31998.0,12,2026,4,25
6,1777205541513,FR,France,CH,Switzerland,2026-04-25,17661.0,12,2026,4,25
7,1777205541513,FR,France,DE,Germany,2026-04-25,15160.0,12,2026,4,25


In [9]:
print("\nGold Daily Mix:")
print_sql_df("SELECT * FROM gold.daily_electricity_mix order by process_ts desc LIMIT 10")


Gold Daily Mix:


,process_ts,zone,zone_name,date,nuclear_pct,biomass_pct,wind_pct,solar_pct,hydro_pct,gas_pct,oil_pct,coal_pct,geothermal_pct,unknown_pct,total_production_mwh,fossil_free_avg_pct,renewable_avg_pct,hours_covered,year,month,day
0,1777206222712,FR,France,2026-04-26,75.668675,2.283793,4.765665,8.373853,8.061745,0.770679,0.07559,0.0,0.0,0.0,604718.204,99.153731,23.485056,12,2026,4,26
1,1777205541513,FR,France,2026-04-25,73.844472,2.209815,5.084784,7.562436,9.987998,1.077564,0.23293,0.0,0.0,0.0,654326.799,98.689506,24.845033,12,2026,4,25


In [10]:
print("\nPipeline State (el_state):")
print_sql_df("SELECT * FROM state.el_state ORDER BY process_ts DESC limit 10")


Pipeline State (el_state):


,process,process_ts,start_timestamp,end_timestamp,status,record_count,error_message
0,gold,1777206222712,2026-04-26 17:53:44.343453+05:30,2026-04-26 17:53:51.542476+05:30,R,8,None
1,silver,1777206207772,2026-04-26 17:53:29.413431+05:30,2026-04-26 17:53:34.413820+05:30,C,24,None
2,bronze,1777205939689,2026-04-26 17:48:59.689627+05:30,2026-04-26 17:52:33.922041+05:30,C,6,None
3,gold,1777205541513,2026-04-26 17:42:23.103938+05:30,2026-04-26 17:42:29.532154+05:30,R,16,None
4,silver,1777205520389,2026-04-26 17:42:01.921156+05:30,2026-04-26 17:42:05.749029+05:30,C,198,None
5,bronze,1777205502572,2026-04-26 17:41:42.572706+05:30,2026-04-26 17:41:45.250530+05:30,C,48,None
